# Plot saved training and test results

Set **RUN_DIR** to one single-training or K-Fold run folder, then run the cells. No training or inference is performed. Use the **rnet** kernel.

Epoch losses and metrics are shown separately for every saved fold, using completed epochs (epoch numbering starts at 1). Boxplots use **postprocessed full-volume** metrics: fixed-split `test` and CV outer `holdout` by default. Native patch metrics are pooled cohort measurements and cannot provide subject boxplots.

Each dot is one animal per condition, modality, region and evaluation role. Repeated predictions of a scan are averaged first; longitudinal scans are then averaged within that animal and modality. Cohorts, roles and units stay separate. Missing annotations and undefined scores are omitted, while genuine zero scores and finite both-empty conventions are retained.

Plots use native Plotly display in VS Code, without widgets or HTML display wrappers. Hover for details, zoom or pan, and click legend entries to hide curves or conditions. Interactive HTML/JSON exports need no additional renderer; optional PNG/PDF/SVG exports require Kaleido.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from datetime import datetime
from importlib.util import find_spec
import hashlib
import json

from IPython.display import display
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go

from src.evaluation.run_plots import animal_metric_points, load_run_plot_data

## 1. Choose a run and plotting settings

Use the run containing `experiment.json` and `fold_0`, rather than a model-group folder. Loss history is read from committed recovery state when available, so interrupted runs are supported.

`METRICS_FILE=None` selects one existing aggregate report from the run, falling back to per-fold evaluation files. To plot a stopped-checkpoint evaluation, set its exact `full_volume_metrics.csv` path explicitly; no latest evaluation is selected. Keep `RUN_DIR` pointing to the associated training run for its loss histories and scan metadata.

Leave filters as `None` to keep all available values. Choose region names from the list printed below. `LOSS_KEYS=None` includes total loss and every saved head loss. Older rat notebook 04 runs store the held-out cohort as `validation`: explicitly select that role for historical runs. New fixed-epoch notebook 04 runs use `holdout`, included by default. This notebook never relabels saved roles.

`HISTORY_METRIC` chooses a metric logged during fitting (for example `regions_logits_mean_dice`, `regions_mean_dice` or `brain_mask_mean_dice`). The matching `val_` curve is included when saved. Leave it as `None` to select the first available metric. This setting is independent of the full-volume boxplot `METRIC`.

In [ ]:
RUN_DIR: Path | None = None  # Example: REPO_ROOT / "results/d1_3drnet_kfold/RUN_..."
METRICS_FILE: Path | None = None  # Optional exact stopped-checkpoint full_volume_metrics.csv.

METRIC: str = "Dice"  # Examples: Dice, HD95, ASSD, Sensitivity, Precision, AVE.
ROLES: tuple[str, ...] = ("test", "holdout")
FOLDS: tuple[int, ...] | None = None
REGIONS: tuple[str, ...] | None = None
DATASETS: tuple[str, ...] | None = None
CONDITIONS: tuple[str, ...] | None = None
MODALITIES: tuple[str, ...] | None = None

LOSS_KEYS: tuple[str, ...] | None = ("loss", "val_loss")
LOSS_LOG_SCALE: bool = False
HISTORY_METRIC: str | None = None  # Choose a key printed below; omit the val_ prefix.
HISTORY_METRIC_LOG_SCALE: bool = False
HISTORY_METRIC_LIMITS: tuple[float, float] | None = None  # For example (0.0, 1.0) for Dice.
SHOW_SUBJECT_POINTS: bool = True
METRIC_LIMITS: tuple[float, float] | None = None  # For example (0.0, 1.0) for Dice.
PLOT_RENDERER: str = "vscode"
EXPORT_SCALE: float = 3.0  # Resolution multiplier for optional static image exports.

EXPORT_RESULTS: bool = False
OUTPUT_DIR: Path | None = None  # None uses RUN_DIR / "plots"; each export creates a new snapshot.
EXPORT_FORMATS: tuple[str, ...] = ("html", "json")  # Optional png/pdf/svg need Kaleido.

## 2. Load saved data and inspect coverage

The table below counts raw saved measurements by annotation/empty-mask status. Inspect coverage before comparing regions: partial annotations can give different animal counts. Missing condition or modality remains `unknown`; known values are joined from saved manifests/contracts without guessing from filenames.

Hover details retain the original ID and explicit strain. If strain is missing, the saved `c52` or `cd1` protocol code is shown. If neither is provided, the value remains `unknown`.

In [ ]:
saved = None
history = pd.DataFrame()
metric_history = pd.DataFrame()
history_metric_name: str | None = None
selected_metrics = pd.DataFrame()
animal_points = pd.DataFrame()
metric_summary = pd.DataFrame()
coverage = pd.DataFrame()
figures: dict[str, go.Figure] = {}

if RUN_DIR is None:
    print("Set RUN_DIR, then rerun this cell and the plotting cells.")
else:
    saved = load_run_plot_data(run_dir=RUN_DIR, metrics_file=METRICS_FILE)
    history = saved.history.copy()
    metric_history = saved.metric_history.copy()
    selected_metrics = saved.metrics.copy()

    if FOLDS is not None and not history.empty:
        history = history.loc[history["Fold"].isin(FOLDS)].copy()

    if FOLDS is not None and not metric_history.empty:
        metric_history = metric_history.loc[metric_history["Fold"].isin(FOLDS)].copy()

    if not metric_history.empty:
        available_history_metrics = sorted({name.removeprefix("val_") for name in metric_history["Metric"].unique()})
        print("Available epoch metrics:", available_history_metrics)

    if not history.empty:
        print("Available loss keys:", sorted(history["Loss"].unique()))
        epoch_counts = history.groupby("Fold")["Epoch"].max().rename("CompletedEpochs")
        display(epoch_counts.to_frame())
    else:
        print("No loss history found for the selected folds.")

    if not selected_metrics.empty:
        print("Available metrics:", sorted(selected_metrics["Metric"].unique()))
        print("Available roles:", sorted(selected_metrics["DatasetRole"].unique()))
        print("Available regions:", sorted(selected_metrics["Region"].unique()))

        filters = {
            "Fold": FOLDS, "Region": REGIONS, "Dataset": DATASETS,
            "Condition": CONDITIONS, "Modality": MODALITIES,
        }
        for column, values in filters.items():
            if values is not None:
                selected_metrics = selected_metrics.loc[selected_metrics[column].isin(values)].copy()

        selection = selected_metrics["Metric"].eq(METRIC) & selected_metrics["DatasetRole"].isin(ROLES)
        selected_metrics = selected_metrics.loc[selection].copy()
        coverage_keys = ["Dataset", "DatasetRole", "Condition", "Modality", "Region", "Status"]
        coverage = selected_metrics.groupby(coverage_keys, dropna=False).agg(
            Animals=("Subject", "nunique"), Scans=("Scan", "nunique"), Rows=("Value", "size"),
        ).reset_index()
        display(coverage)
    else:
        print("No full-volume report found. Loss plots remain available; evaluate saved weights or set METRICS_FILE.")

## 3. Training and validation loss curves

Every panel represents one fold. Curves retain their actual completed duration; shorter folds are not padded or extrapolated. Change `LOSS_KEYS` to inspect ROI, brain or classifier losses, whose scales may differ. No smoothing is applied.

Hover shows the epoch and loss values. Click a legend entry to toggle a curve, or double-click to isolate it.

In [ ]:
if not history.empty:
    loss_history = history.copy()
    if LOSS_KEYS is not None:
        loss_history = loss_history.loc[loss_history["Loss"].isin(LOSS_KEYS)].copy()
    if loss_history.empty:
        raise ValueError("No saved loss matches LOSS_KEYS; inspect the available keys above")

    loss_history = loss_history.sort_values(["Fold", "Loss", "Epoch"])
    fold_ids = sorted(loss_history["Fold"].unique())
    columns = min(2, len(fold_ids))
    rows = (len(fold_ids) + columns - 1) // columns
    figure_height = 360 * rows
    loss_dashes: dict[str, str] = {}
    for name in loss_history["Loss"].unique():
        style = "solid"
        if name.startswith("val_"):
            style = "dash"
        loss_dashes[name] = style

    loss_figure = px.line(
        data_frame=loss_history, x="Epoch", y="Value", color="Loss", line_dash="Loss",
        line_dash_map=loss_dashes, facet_col="Fold", facet_col_wrap=columns,
        category_orders={"Fold": fold_ids}, labels={"Value": "Loss"},
        title="Saved training and validation losses", template="plotly_white",
        height=figure_height, log_y=LOSS_LOG_SCALE,
    )
    loss_figure.update_layout(hovermode="x unified")
    loss_figure.update_xaxes(title_text="Epoch", matches=None)
    figures["loss_curves"] = loss_figure
    display_options = {"scrollZoom": True, "displaylogo": False}
    loss_figure.show(renderer=PLOT_RENDERER, config=display_options)

## 4. Training and validation metric curves

Select `HISTORY_METRIC` from the available epoch metrics printed above. Every panel represents one fold, with solid training and dashed validation curves, the actual saved epochs, and interactive hover/zoom. Change `HISTORY_METRIC_LIMITS` to set the y-axis range.

These are the native metrics logged on training/validation patches during fitting. Postprocessed full-volume test metrics are evaluated separately and cannot supply epoch curves. Missing validation observations remain gaps; runs without saved validation metrics show only training. No smoothing, interpolation or extra evaluation is performed.

In [ ]:
if not metric_history.empty:
    available_history_metrics = sorted({name.removeprefix("val_") for name in metric_history["Metric"].unique()})
    history_metric_name = HISTORY_METRIC
    if history_metric_name is None:
        history_metric_name = available_history_metrics[0]
    history_metric_name = history_metric_name.removeprefix("val_")
    validation_key = "val_" + history_metric_name
    curve_keys = (history_metric_name, validation_key)
    selected_history = metric_history.loc[metric_history["Metric"].isin(curve_keys)].copy()
    if selected_history.empty:
        raise ValueError(f"HISTORY_METRIC is not saved. Choose from {available_history_metrics}")

    selected_history = selected_history.sort_values(["Fold", "Metric", "Epoch"])
    fold_ids = sorted(selected_history["Fold"].unique())
    columns = min(2, len(fold_ids))
    rows = (len(fold_ids) + columns - 1) // columns
    figure_height = 360 * rows
    curve_styles = {history_metric_name: "solid", validation_key: "dash"}
    category_orders = {"Fold": fold_ids, "Metric": list(curve_keys)}
    figure_title = f"Training and validation: {history_metric_name}"

    metric_curve_figure = px.line(
        data_frame=selected_history, x="Epoch", y="Value", color="Metric", line_dash="Metric",
        line_dash_map=curve_styles, facet_col="Fold", facet_col_wrap=columns,
        category_orders=category_orders, labels={"Value": history_metric_name},
        title=figure_title, template="plotly_white", height=figure_height,
        log_y=HISTORY_METRIC_LOG_SCALE, range_y=HISTORY_METRIC_LIMITS,
    )
    metric_curve_figure.update_layout(hovermode="x unified")
    metric_curve_figure.update_xaxes(title_text="Epoch", matches=None)
    figures["metric_curves"] = metric_curve_figure
    display_options = {"scrollZoom": True, "displaylogo": False}
    metric_curve_figure.show(renderer=PLOT_RENDERER, config=display_options)
elif saved is not None:
    print("No per-epoch metrics were saved for the selected folds; loss curves remain available.")

## 5. Chosen test metric by condition and modality

Regions are on the x-axis, condition sets the color, and modalities have separate columns. Rows separate dataset, evaluation role and unit. Dots and boxplots use the same animal averages. An animal appearing in several modalities contributes once to each modality, rather than being treated as several independent animals in a pooled score.

The accompanying summary is descriptive: mean, SD, median and range across the plotted animals. It is not a confidence interval or an average of fold averages. `Folds` in the points table records how many models contributed to an animal's selected measurements.

Hover over a point to see the normalized animal ID, original ID, strain and contributing scan/fold counts. Legend entries toggle SHAM/TBI visibility. The y-axis is shared across modalities within a row; separate cohort/role/unit rows retain independent scales.

In [ ]:
if saved is not None and not saved.metrics.empty:
    animal_points = animal_metric_points(metrics=selected_metrics, metric=METRIC, roles=ROLES)
    summary_keys = ["Dataset", "DatasetRole", "Condition", "Modality", "Head", "Region", "Metric", "Unit"]
    metric_summary = animal_points.groupby(summary_keys, dropna=False).agg(
        Animals=("Subject", "nunique"), Scans=("Scans", "sum"),
        Mean=("Value", "mean"), SD=("Value", "std"), Median=("Value", "median"),
        Minimum=("Value", "min"), Maximum=("Value", "max"),
    ).reset_index()
    display(metric_summary)

    plot_data = animal_points.copy()
    plot_data["Panel"] = plot_data["Dataset"] + " / " + plot_data["DatasetRole"] + " / " + plot_data["Unit"]
    region_order = list(plot_data["Region"].unique())
    if REGIONS is not None:
        region_order = list(REGIONS)
    condition_order = sorted(plot_data["Condition"].unique())
    modality_order = sorted(plot_data["Modality"].unique())
    panel_order = list(plot_data["Panel"].unique())
    category_orders = {
        "Region": region_order, "Condition": condition_order,
        "Modality": modality_order, "Panel": panel_order,
    }
    hover_columns = ["OriginalID", "Strain", "Dataset", "DatasetRole", "Modality", "Scans", "Folds"]
    figure_height = 420 * len(panel_order)
    facet_width = max(520, len(region_order) * 90)
    figure_width = facet_width * len(modality_order)
    points_mode = "outliers"
    if SHOW_SUBJECT_POINTS:
        points_mode = "all"

    metric_figure = px.box(
        data_frame=plot_data, x="Region", y="Value", color="Condition",
        facet_row="Panel", facet_col="Modality", category_orders=category_orders,
        color_discrete_sequence=px.colors.qualitative.Safe, points=points_mode,
        hover_name="Subject", hover_data=hover_columns,
        labels={"Value": METRIC, "OriginalID": "Original ID"},
        title=f"{METRIC}: full-volume animal means", template="plotly_white",
        height=figure_height, width=figure_width,
    )
    metric_figure.update_layout(boxmode="group")
    metric_figure.update_xaxes(title_text="Region", tickangle=-45, matches=None)
    metric_figure.update_yaxes(matches=None, range=METRIC_LIMITS)
    for row in range(1, len(panel_order) + 1):
        first_axis = (row - 1) * len(modality_order) + 1
        reference_axis = "y"
        if first_axis > 1:
            reference_axis = f"y{first_axis}"
        for column in range(2, len(modality_order) + 1):
            metric_figure.update_yaxes(matches=reference_axis, row=row, col=column)

    figures["metric_boxplots"] = metric_figure
    display_options = {"scrollZoom": True, "displaylogo": False}
    metric_figure.show(renderer=PLOT_RENDERER, config=display_options)

## 6. Export figures and their exact plotting data

Set `EXPORT_RESULTS=True` after inspecting the plots. A new snapshot contains interactive HTML/JSON figures, loss/metric epoch observations, selected raw measurements (including missing statuses), animal points, summary/coverage CSVs and a JSON record of settings and input SHA256 hashes. HTML files embed Plotly and can be opened offline. Existing results are not overwritten.

For manuscript images, select `EXPORT_FORMATS=("png", "pdf", "svg")` with a compatible Kaleido installation. PDF/SVG are vector exports; `EXPORT_SCALE` controls static image resolution. Native Plotly display and HTML/JSON exports work without Kaleido.

In [ ]:
if EXPORT_RESULTS:
    if saved is None or not figures:
        raise ValueError("Select a run and create at least one figure before exporting")
    allowed_formats = {"html", "json", "png", "pdf", "svg"}
    if not EXPORT_FORMATS or not set(EXPORT_FORMATS).issubset(allowed_formats):
        raise ValueError("Choose EXPORT_FORMATS from html, json, png, pdf and svg")
    static_formats = {"png", "pdf", "svg"}.intersection(EXPORT_FORMATS)
    if static_formats and find_spec("kaleido") is None:
        raise ValueError("PNG/PDF/SVG export requires compatible Kaleido; use html/json without it")
    if EXPORT_SCALE <= 0:
        raise ValueError("EXPORT_SCALE must be positive")

    export_root = OUTPUT_DIR
    if export_root is None:
        export_root = RUN_DIR / "plots"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    export_dir = export_root / timestamp
    export_dir.mkdir(parents=True, exist_ok=False)

    for name, figure in figures.items():
        for extension in EXPORT_FORMATS:
            destination = export_dir / f"{name}.{extension}"
            if extension == "html":
                figure.write_html(file=destination, include_plotlyjs=True, full_html=True)
            elif extension == "json":
                figure.write_json(file=destination)
            else:
                figure.write_image(file=destination, format=extension, scale=EXPORT_SCALE)

    export_tables = {
        "loss_history": history, "metric_history": metric_history,
        "selected_scan_metrics": selected_metrics,
        "animal_metric_points": animal_points, "metric_summary": metric_summary,
        "annotation_coverage": coverage,
    }
    for name, table in export_tables.items():
        destination = export_dir / f"{name}.csv"
        table.to_csv(destination, index=False)

    input_records: list[dict[str, str]] = []
    for path in saved.sources:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        input_records.append({"Path": str(path), "SHA256": digest})
    settings = {
        "run_dir": str(RUN_DIR.resolve()), "metrics_file": METRICS_FILE,
        "metric": METRIC, "roles": ROLES, "folds": FOLDS, "regions": REGIONS,
        "datasets": DATASETS, "conditions": CONDITIONS, "modalities": MODALITIES,
        "loss_keys": LOSS_KEYS, "loss_log_scale": LOSS_LOG_SCALE,
        "history_metric": history_metric_name, "history_metric_log_scale": HISTORY_METRIC_LOG_SCALE,
        "history_metric_limits": HISTORY_METRIC_LIMITS,
        "show_subject_points": SHOW_SUBJECT_POINTS, "metric_limits": METRIC_LIMITS,
        "formats": EXPORT_FORMATS, "export_scale": EXPORT_SCALE,
        "plot_backend": "plotly", "plotly_version": plotly.__version__, "renderer": PLOT_RENDERER,
        "aggregation": "mean repeated predictions per scan, then mean scans per animal within modality/condition/region/role",
        "sources": input_records,
    }
    settings_file = export_dir / "plot_settings.json"
    settings_file.write_text(json.dumps(settings, indent=2, default=str), encoding="utf-8")
    print("Exported figures and plotting data to:", export_dir)
else:
    print("Set EXPORT_RESULTS=True to export the displayed figures and data.")